In [1]:
#ERA5 Parse Data
import cdsapi

#https://cds.climate.copernicus.eu/datasets/reanalysis-era5-single-levels-timeseries?tab=download

dataset = "reanalysis-era5-single-levels-timeseries"
request = {
    "variable": [
        "2m_dewpoint_temperature",
        "surface_pressure",
        "2m_temperature",
        "total_precipitation",
        "10m_u_component_of_wind",
        "10m_v_component_of_wind"
    ],
    "location": {"longitude": 48, "latitude": 20.5},
    "date": ["1940-01-01/2026-02-07"],
    "data_format": "netcdf"
 #   "data_format": "csv"
}

client = cdsapi.Client()
client.retrieve(dataset, request).download("era5_data_fix.zip")



2026-02-13 14:09:00,483 WARNING [2025-03-17T00:00:00] Please be aware that the generation of this dataset is using an alternative source for the ERA5 data and may be subject to changes over time (e.g. file format, data file structure, deprecation etc). This dataset should therefore be regarded as “experimental” and is **not recommended for use in a production environment**. 

Notification of changes via this catalogue entry banner and/or in the [Forum](https://forum.ecmwf.int/) will be provided on best efforts.
2026-02-13 14:09:00,484 INFO Request ID is 0088f445-6250-4f9c-8b17-c2dbfa92f48e
2026-02-13 14:09:00,572 INFO status has been updated to accepted
2026-02-13 14:09:09,024 INFO status has been updated to running
2026-02-13 14:09:21,832 INFO status has been updated to successful


b42c5a60fddbdb64ba13cfa2a3dc08a4.zip:   0%|          | 0.00/23.1M [00:00<?, ?B/s]

'era5_data_fix.zip'

In [4]:
# 2. NetCDF beolvasása 
from dotenv import load_dotenv
import os
from sqlalchemy import create_engine
import pandas as pd
import shutil
import xarray as xr
import zipfile

load_dotenv() # betölti a .env fájlt

db_host = os.getenv("DB_HOST")
db_user = os.getenv("DB_USER")
db_password = os.getenv("DB_PASSWORD")
db_name = os.getenv("DB_NAME")

engine = create_engine( f"mysql+pymysql://{db_user}:{db_password}@{db_host}/{db_name}")

zip_path = "era5_data_fix.zip"
extract_dir = "extracted-fix"

# 1. ZIP kibontása
with zipfile.ZipFile(zip_path, "r") as z:
    z.extractall(extract_dir)

# 2. Összegyűjtjük az összes .nc fájlt
nc_files = [os.path.join(extract_dir, f) for f in os.listdir(extract_dir) if f.endswith(".nc")]

if not nc_files:
    raise FileNotFoundError("Nem található .nc fájl a ZIP-ben!")

print("Megtalált NetCDF fájlok:")
for f in nc_files:
    print(" -", f)

# 3. Mindegyiket beolvassuk és DataFrame-re alakítjuk
dfs = []
for nc in nc_files:
    print("Beolvasás:", nc)
    ds = xr.open_dataset(nc)
    df = ds.to_dataframe().reset_index()
    dfs.append(df)

# 3. NetCDF-ek összeolvasztása
datasets = [xr.open_dataset(nc) for nc in nc_files]
ds_merged = xr.merge(datasets, compat="override")

# 4. DataFrame konverzió
df = ds_merged.to_dataframe().reset_index()

# --- SZÁMOLT ÉRTÉKEK HOZZÁADÁSA ---

import numpy as np

df["temp_c"] = df["t2m"] - 273.15
df["dewpoint_c"] = df["d2m"] - 273.15

def saturation_vapor_pressure(T):
    return 6.112 * np.exp((17.67 * T) / (T + 243.5))

e_t = saturation_vapor_pressure(df["temp_c"])
e_td = saturation_vapor_pressure(df["dewpoint_c"])

df["rh"] = 100 * (e_td / e_t)
df["rh"] = df["rh"].clip(0, 100)

df["wind_speed"] = np.sqrt(df["u10"]**2 + df["v10"]**2)

df["wind_dir"] = np.degrees(np.arctan2(df["u10"], df["v10"]))
df["wind_dir"] = (df["wind_dir"] + 360) % 360

# 5. Típusok normalizálása
df = df.convert_dtypes()
df = df.where(df.notnull(), None)

# 6. Betöltés MySQL-be
df.to_sql("era5_adatok_fix", engine, if_exists="replace", index=False, chunksize=5000)

print("Sikeresen betöltve!")


Megtalált NetCDF fájlok:
 - extracted-fix/reanalysis-era5-single-levels-timeseries-sfc5kmidow6.nc
Beolvasás: extracted-fix/reanalysis-era5-single-levels-timeseries-sfc5kmidow6.nc
Sikeresen betöltve!
